<a href="https://colab.research.google.com/github/sayandeepmaity/luminator/blob/main/predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import pandas as pd
import numpy as np

# Number of test samples you want to generate
num_samples = 100

# Base features used by Model 1 and Model 2
features = [
    'correlation', 'phase', 'amplitude', 'energy', 'spectral_centroid', 'snr',
    'tau_01', 'tau_02', 'tau_03', 'tau_04', 'tau_05',
    'tau_12', 'tau_13', 'tau_14', 'tau_15', 'tau_23', 'tau_24', 'tau_25',
    'tau_34', 'tau_35', 'tau_45',
    'corr_01', 'corr_02', 'corr_03', 'corr_04', 'corr_05',
    'corr_12', 'corr_13', 'corr_14', 'corr_15',
    'corr_23', 'corr_24', 'corr_25', 'corr_34', 'corr_35', 'corr_45',
    'snr_01', 'snr_02', 'snr_03', 'snr_04', 'snr_05',
    'snr_12', 'snr_13', 'snr_14', 'snr_15',
    'snr_23', 'snr_24', 'snr_25', 'snr_34', 'snr_35', 'snr_45'
]

# Create random but realistic values for each feature
data = {feature: np.random.rand(num_samples) * np.random.uniform(1, 10) for feature in features}

# Create the DataFrame
df = pd.DataFrame(data)

# Save to CSV with the updated file path
df.to_csv('/content/drive/MyDrive/gunshotdata/testgundata.csv', index=False)

print("✅ testgundata.csv created with shape:", df.shape)


✅ testgundata.csv created with shape: (100, 51)


In [31]:
import pandas as pd
import joblib
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


# Load the trained models
model_gunshot = joblib.load('/content/drive/My Drive/gunshotdata/dmodel1/gunshot_classifier_model.pkl')  # Gunshot detection model
model_location = joblib.load('/content/drive/My Drive/gunshotdata/dmodel2/model2_tdoaselector.pkl')  # Best pair model (removed in this version)
model3 = joblib.load('/content/drive/My Drive/gunshotdata/dmodel3/gun_type_model.pkl')  # Gun type prediction model

# Load the test data
df = pd.read_csv('/content/drive/MyDrive/gunshotdata/testgundata.csv')  # Test dataset

# Select only the required features for Model 1 (6 features)
required_features_model1 = ['correlation', 'phase', 'amplitude', 'energy', 'spectral_centroid', 'snr']
df_model1 = df[required_features_model1]

# Initialize a list to store results for each sample
results = []

# Loop through each sample in the dataset and predict step by step
for idx, row in df_model1.iterrows():
    # Model 1: Predict whether the event is a gunshot or not
    gunshot_pred = model_gunshot.predict([row])[0]  # Predict for a single sample

    if gunshot_pred == 1:
        # Gunshot detected, process Model 3 for gun type prediction
        # We need to use the full set of features that Model 3 expects, so pass the entire row from the original df
        row_model3 = df.loc[idx, :]  # Use the original row with all features for Model 3

        # Ensure only the 49 features that Model 3 expects are passed
        required_features_model3 = ['amplitude', 'energy', 'spectral_centroid', 'snr',
                                    'tau_01', 'tau_02', 'tau_03', 'tau_04', 'tau_05', 'tau_12', 'tau_13', 'tau_14',
                                    'tau_15', 'tau_23', 'tau_24', 'tau_25', 'tau_34', 'tau_35', 'tau_45', 'corr_01',
                                    'corr_02', 'corr_03', 'corr_04', 'corr_05', 'corr_12', 'corr_13', 'corr_14',
                                    'corr_15', 'corr_23', 'corr_24', 'corr_25', 'corr_34', 'corr_35', 'corr_45',
                                    'snr_01', 'snr_02', 'snr_03', 'snr_04', 'snr_05', 'snr_12', 'snr_13', 'snr_14',
                                    'snr_15', 'snr_23', 'snr_24', 'snr_25', 'snr_34', 'snr_35', 'snr_45']

        # Filter the row to match Model 3's expected features (49 features)
        row_model3_filtered = row_model3[required_features_model3]

        # Predict gun type (pistol or rifle) - Use the filtered row with the 49 features Model 3 expects
        gun_type_pred = model3.predict([row_model3_filtered])[0]

        # Store the result for this sample
        results.append({
            'sample_index': idx,
            'gunshot_detected': True,
            'gun_type': gun_type_pred
        })

        print(f"✅ Sample {idx}: Gunshot detected!")
        print(f"   Gun Type: {gun_type_pred}")

    else:
        # No gunshot detected
        results.append({
            'sample_index': idx,
            'gunshot_detected': False,
            'gun_type': None
        })

        print(f"❌ Sample {idx}: No gunshot detected.")

# Convert the results to a DataFrame for easy viewing
results_df = pd.DataFrame(results)

# Save the results to a CSV file
results_df.to_csv('/content/drive/MyDrive/gunshotdata/gunshot_predictions_without_bestpair.csv', index=False)

print("✅ All predictions saved to 'gunshot_predictions_without_bestpair.csv'.")


✅ Sample 0: Gunshot detected!
   Gun Type: Pistol
✅ Sample 1: Gunshot detected!
   Gun Type: Rifle
✅ Sample 2: Gunshot detected!
   Gun Type: Pistol
✅ Sample 3: Gunshot detected!
   Gun Type: Pistol
✅ Sample 4: Gunshot detected!
   Gun Type: Pistol
✅ Sample 5: Gunshot detected!
   Gun Type: Pistol
✅ Sample 6: Gunshot detected!
   Gun Type: Pistol
✅ Sample 7: Gunshot detected!
   Gun Type: Pistol
✅ Sample 8: Gunshot detected!
   Gun Type: Pistol
✅ Sample 9: Gunshot detected!
   Gun Type: Pistol
✅ Sample 10: Gunshot detected!
   Gun Type: Pistol
✅ Sample 11: Gunshot detected!
   Gun Type: Rifle
✅ Sample 12: Gunshot detected!
   Gun Type: Pistol
✅ Sample 13: Gunshot detected!
   Gun Type: Pistol
✅ Sample 14: Gunshot detected!
   Gun Type: Pistol
✅ Sample 15: Gunshot detected!
   Gun Type: Pistol
❌ Sample 16: No gunshot detected.
✅ Sample 17: Gunshot detected!
   Gun Type: Pistol
✅ Sample 18: Gunshot detected!
   Gun Type: Rifle
✅ Sample 19: Gunshot detected!
   Gun Type: Rifle
✅ Sample 20